In [ ]:
import torch
import torch.nn as nn
from scipy.io import wavfile
from scipy import signal
import numpy as np
import glob
import sys
import os
import tempfile

import matplotlib.pyplot as plt
sys.path.append('../utls/')
sys.path.append('../src/model/')

sys.path.append('/om2/user/bjmedina/')

import DistanceMemoryModel
import encoders


from chexture_choolbox.auditorytexture.statistics_sets import (
    STAT_SET_FULL_MCDERMOTTSIMONCELLI as statistics_dict
)
from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params

from sklearn.decomposition import PCA

sys.path.append('/om/user/jmhicks/projects/TextureSimilarity/code/')
import texture_similarity.utils as ts


import scipy.signal
import scipy.signal.windows

if not hasattr(scipy.signal, "kaiser"):
    scipy.signal.kaiser = scipy.signal.windows.kaiser


In [ ]:
ARCHITECTURES =  ['kell2018', 'resnet50']
TASKS = ['word', 'speaker', 'audioset', 'word_speaker_audioset']
LAYERS = {
    'kell2018': [
        'input_after_preproc', 'relu0', 'relu1', 'relu2', 'relu3', 'relu4', 'relufc'
    ],
    'resnet50': [
        'input_after_preproc', 'conv1_relu1', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool'
    ],
}

networks = []
for architecture in ARCHITECTURES:
    for task in TASKS:
        for layer in LAYERS[architecture]:
            networks.append({'name': f'{architecture}_{task}',
                            'layer': layer})


model_name = ARCHITECTURES[0]
layer = LAYERS[model_name][3]
task  = TASKS[3]

sys.path.append(f'/om2/user/bjmedina/models/cochdnn/model_directories/{model_name}_{task}/')
print(f'/om2/user/bjmedina/models/cochdnn/model_directories/{model_name}_{task}/')


layer_model = encoders.Kell2018Encoder(
    model_name=model_name,
    layer=layer,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device='cuda'
)

In [ ]:
# grabbing example list of sound
sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/*wav")
texture_list = sounds_list

ALL_SOUNDS = glob.glob("/om2/data/public/audioset/wavs/unbalanced_train_segments_downloads/unbalanced_train_segments_downloads_*/*wav")
# print(len(ALL_SOUNDS))

In [ ]:
encoding = layer_model(ALL_SOUNDS[0])

print(encoding)

In [ ]:
# #features = ts.features.extract_from_cochdnn(model_name, layer, sounds_list)

# test_sr = 20000
# test_duration = 2.0
# target_len         = int(test_sr * test_duration)  # expected length in samples
# original_sr, sound = wavfile.read(ALL_SOUNDS[0])
# if sound.ndim > 1:
#     sound = sound.mean(axis=1)

# # Resample
# sound = signal.resample_poly(sound, test_sr, original_sr, axis=0)

#  # Check length
# if len(sound) < target_len:
#     print("too short")
#     # Drop short sounds
#     raise ValueError(f"Sound too short after resampling: {len(sound)} < {target_len}")
# elif len(sound) > target_len:
#     print("long...")

#     # Random crop
#     start = np.random.randint(0, len(sound) - target_len + 1)
#     sound = sound[start:start + target_len]

#     # Zero-mean and RMS normalize to target
#     sound = sound - np.mean(sound)
#     rms = np.sqrt(np.mean(sound ** 2))
#     sound = 0.05 * sound / (rms + 1e-6)

# def _write_temp_wav(y):
#     import tempfile
#     """Write a temp WAV file at self.sr from float signal y; returns path. Caller must remove."""
#     # Convert to 32-bit float WAV for fidelity
#     tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#     tmp_path = tmp.name
#     tmp.close()
#     # wavfile.write expects int or float; float32 is standard for PCM-float
#     wavfile.write(tmp_path, test_sr, y.astype(np.float32))
#     return tmp_path

# tmp_fn = _write_temp_wav(sound)
# print(tmp_fn)
# features = ts.features.extract_from_cochdnn(model_name, layer, [tmp_fn])
# #features = ts.features.extract_from_cochdnn(model_name, layer, sound)

In [ ]:
features_dict = {}
for layer in LAYERS[model_name]:
    print(layer)
    layer_model = encoders.Kell2018Encoder(
    model_name=model_name,
    layer=layer,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device='cuda'
    )

    # zscore_projector = encoders.ZScoreSpace(layer_model, device='cuda')
    # zscore_projector.fit(texture_list)
    # clear_output(wait=False)

    features = []
    for sound in ALL_SOUNDS[:100]:
        feature = layer_model(sound)
        features.append(feature)

    features = torch.stack(features, dim=0)
    features_dict[layer] = features
    print(features.shape)

In [ ]:
plt.hist(features_dict['input_after_preproc'].cpu(), bins=100)

In [ ]:
print(sounds_list)